# CWK1 - S&P 500 weekly return, beta, adapted RFM and clustering (corrected notebook)

Study period 1 Jan 2021 to 1 Jan 2025 (end exclusive). Same structure as the original notebook.

**Fixes made**
1. Duplicate `weekly_return` cell removed (it overwrote the version with the first blank week dropped).
2. Observation date is derived from the data, not typed in.
3. One seed and one `N_INIT` used for every K-means fit.
4. VIF replaced with the standard formula (inverse correlation matrix).
5. Validation rules vectorised; `pct_change(fill_method=None)` replaced by `x / x.shift(1) - 1` (works on every pandas version).

**Added (required by the brief)**
corporate-action check, survivorship evidence, no-event count, worked RFM example, unit tests and alignment checks, final inclusion/exclusion log, cluster profiles, adjusted Rand index, seed stability, and section 2.5.2 (removal of Monetary).

## 1.1 Data ingestion

In [ ]:
import os, datetime as dt, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import StringIO
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

start = "2021-01-01"
end   = "2025-01-01"                 # end date is exclusive
START_TS, END_TS = pd.Timestamp(start), pd.Timestamp(end)
BENCH_SYMBOL = "^GSPC"

CACHE_DIR = "cache"; OUT_DIR = "outputs"
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# package versions (required by the brief) - keep this output in the submission
import sys, platform, importlib
print("python", sys.version.split()[0], "|", platform.platform())
for m_ in ["pandas", "numpy", "scipy", "sklearn", "matplotlib", "seaborn", "yfinance", "requests"]:
    try: print(f"{m_:12s}", importlib.import_module(m_).__version__)
    except Exception: print(f"{m_:12s} not installed")

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

retrieval_ts   = dt.datetime.now()                 # retrieval date (required by the brief)
retrieval_date = retrieval_ts.date().isoformat()

resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
resp.raise_for_status()
sp500_df = pd.read_html(StringIO(resp.text))[0]

# Yahoo uses "-" where Wikipedia uses "." (BRK.B -> BRK-B)
tickers = sp500_df["Symbol"].str.replace(".", "-", regex=False).tolist()

sp500_df["Retrieved_On"] = retrieval_ts.strftime("%Y-%m-%d %H:%M:%S")
sp500_df.to_csv(f"{CACHE_DIR}/sp500_constituents_{retrieval_date}.csv", index=False)

print("Retrieval date    :", retrieval_date)
print("Tickers extracted :", len(tickers))
sp500_df[["Symbol", "Security", "GICS Sector"]].head(10)

In [ ]:
# Provider: yfinance (free, no API key, returns Close, Adj Close and corporate actions together)
all_symbols = tickers + [BENCH_SYMBOL]            # shares + benchmark in one download

raw_df = yf.download(
    all_symbols, start=start, end=end,
    auto_adjust=False,      # keep both Close and Adj Close
    actions=True,           # keep Dividends and Stock Splits
    progress=False
)
print("Shape   :", raw_df.shape)
raw_df.head()

In [ ]:
# wide (field, ticker) -> one long table; pd.concat keeps all-missing rows so gaps stay visible
present = raw_df.columns.get_level_values(1).unique()
df_all = (
    pd.concat({t: raw_df.xs(t, axis=1, level=1) for t in present}, names=["Ticker", "Date"])
    .reset_index()
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)
df_all["Date"] = pd.to_datetime(df_all["Date"]).dt.tz_localize(None)
df_all.to_csv("df_all_sp500_2021_2025.csv", index=False)          # cached dataset
print("Saved:", os.path.abspath("df_all_sp500_2021_2025.csv"))
df_all.head()

If the live download fails, skip the three cells above and load the cached file instead (same period only; any other period needs tutor agreement):

`df_all = pd.read_csv("df_all_sp500_2021_2025.csv", parse_dates=["Date"])`

## 1.2 Data validation and preprocessing

In [ ]:
PRICE_VOL = ["Adj Close", "Close", "Volume"]

# 1. duplicate dates
dup_rows = df_all.duplicated(subset=["Ticker", "Date"]).sum()
print("1. Duplicate (Ticker, Date) rows:", dup_rows)
if dup_rows:
    print(df_all[df_all.duplicated(subset=["Ticker", "Date"], keep=False)].head())

# 2. missing values
missing = df_all.groupby("Ticker")[PRICE_VOL].apply(lambda g: g.isna().sum())
print("\n2. Tickers with missing values:", int((missing.sum(axis=1) > 0).sum()))
print(missing[missing.sum(axis=1) > 0].sort_values("Adj Close", ascending=False).head(20))

# 3. non-positive prices
nonpos = df_all[(df_all["Adj Close"] <= 0) | (df_all["Close"] <= 0)]
print("\n3. Rows with non-positive Adj Close / Close:", len(nonpos))

# 4. volume errors
vol_neg  = df_all[df_all["Volume"] < 0]
vol_zero = df_all[df_all["Volume"] == 0]
vol_frac = df_all[df_all["Volume"].notna() & (df_all["Volume"] % 1 != 0)]
print("\n4. Volume errors -> negative:", len(vol_neg), "| zero:", len(vol_zero), "| non-integer:", len(vol_frac))
print("   Tickers with zero-volume days:", vol_zero["Ticker"].nunique())

In [ ]:
# 5. unequal histories against the benchmark trading calendar
bench  = df_all[df_all["Ticker"] == BENCH_SYMBOL]
shares = df_all[df_all["Ticker"] != BENCH_SYMBOL]

calendar = pd.DatetimeIndex(bench.dropna(subset=["Adj Close"])["Date"].unique()).sort_values()

hist = (shares.dropna(subset=["Adj Close"]).groupby("Ticker")["Date"]
        .agg(first="min", last="max", n="count"))
hist["expected"]     = len(calendar)
hist["missing_days"] = hist["expected"] - hist["n"]
hist["late_start"]   = hist["first"] > calendar[0]
hist["early_end"]    = hist["last"]  < calendar[-1]
n_no_data = shares["Ticker"].nunique() - len(hist)

print(f"5. Benchmark trading days: {len(calendar)} ({calendar[0].date()} to {calendar[-1].date()})")
print("   Tickers with a full history  :", int((hist["missing_days"] == 0).sum()))
print("   Tickers with missing days    :", int((hist["missing_days"] > 0).sum()))
print("   Late starts / early ends     :", int(hist["late_start"].sum()), "/", int(hist["early_end"].sum()))
print("   Tickers with NO data at all  :", n_no_data)
hist[hist["missing_days"] > 0].sort_values("missing_days", ascending=False).head(20)

In [ ]:
# 6. corporate actions (kept, not excluded: Adj Close already accounts for them)
splits = shares[shares["Stock Splits"].fillna(0) > 0][["Ticker", "Date", "Stock Splits"]]
divs   = shares[shares["Dividends"].fillna(0) > 0].groupby("Ticker").size()
print("6. Split events:", len(splits), "across", splits["Ticker"].nunique(), "tickers")
print(splits.sort_values("Date").to_string(index=False))
print("   Tickers paying dividends in the period:", len(divs))

# large one-day Close moves with no recorded split = possible unrecorded action or bad data
s = shares.sort_values(["Ticker", "Date"]).copy()
s["close_ret"] = s["Close"] / s.groupby("Ticker")["Close"].shift(1) - 1
jumps = s[(s["close_ret"].abs() > 0.35) & (s["Stock Splits"].fillna(0) == 0)]
print("   Unexplained >35% one-day Close moves:", len(jumps))
print(jumps[["Ticker", "Date", "close_ret"]].head(10).to_string(index=False))

# splits: Yahoo Close and Volume are split-adjusted consistently, so Close x Volume (Monetary) stays valid

In [ ]:
# survivorship-bias evidence: how many current constituents joined after the study start?
if "Date added" in sp500_df.columns:
    added = pd.to_datetime(sp500_df["Date added"], errors="coerce")
    print(f"Constituents added after {start}: {(added > START_TS).sum()} of {len(added)}")
    print("Historical members that left the index before retrieval are NOT in the list at all.")

In [ ]:
# ---- exclusion rule: every benchmark trading day must have Adj Close, Close and Volume; prices > 0; volume > 0 ----
n_days = len(calendar)
all_t  = shares["Ticker"].unique()

good_days    = shares.dropna(subset=PRICE_VOL).groupby("Ticker").size().reindex(all_t).fillna(0).astype(int)
nonpos_price = ((shares["Adj Close"] <= 0) | (shares["Close"] <= 0)).groupby(shares["Ticker"]).sum().reindex(all_t)
bad_volume   = (shares["Volume"] <= 0).groupby(shares["Ticker"]).sum().reindex(all_t)

check = pd.DataFrame({"good_days": good_days, "nonpos_price": nonpos_price, "bad_volume": bad_volume})
check.index.name = "Ticker"

def reason(r):
    why = []
    if r.good_days < n_days: why.append(f"incomplete history ({r.good_days} of {n_days} days)")
    if r.nonpos_price > 0:   why.append("non-positive price")
    if r.bad_volume > 0:     why.append("zero or negative volume")
    return "; ".join(why)

check["Reason"] = check.apply(reason, axis=1)
check["Status"] = np.where(check["Reason"] == "", "Included", "Excluded")

keep = check.index[check["Status"] == "Included"].tolist()
df_clean = df_all[df_all["Ticker"].isin(keep + [BENCH_SYMBOL])].copy()

print("Shares downloaded :", len(check))
print("Shares excluded   :", (check["Status"] == "Excluded").sum())
print("Shares retained   :", len(keep))
print("\nExcluded shares and reasons:")
print(check.loc[check["Status"] == "Excluded", "Reason"].to_string())

check.to_csv(f"{OUT_DIR}/inclusion_exclusion_log.csv")
df_clean.to_csv("df_clean.csv", index=False)
df_clean.shape

#### Corporate actions: diagnostic and infographic

In [ ]:
# ---------- corporate actions: diagnostic + infographic (retained shares only) ----------
import matplotlib.gridspec as gridspec

# 1. why might the dividend check show nothing? look at the raw column first
print("Dividends column present   :", "Dividends" in df_all.columns)
print("Dividends non-missing rows :", int(df_all["Dividends"].notna().sum()), "| rows > 0:", int((df_all["Dividends"].fillna(0) > 0).sum()))

r = df_all[df_all["Ticker"].isin(keep)]                         # retained shares only
div_ev = r[r["Dividends"].fillna(0) > 0]
spl_ev = r[r["Stock Splits"].fillna(0) > 0]
n_ret, payers = len(keep), div_ev["Ticker"].nunique()
n_jump = int(jumps["Ticker"].isin(keep).sum())
print(f"Dividend events: {len(div_ev)} | payers: {payers} | total per share: ${div_ev['Dividends'].sum():,.2f} | split events: {len(spl_ev)}")
if len(div_ev) == 0:
    print("-> No dividend rows found. Re-run the download with actions=True and auto_adjust=False, then rebuild df_all.")

# 2. infographic
C = {"div": "#534AB7", "ok": "#1D9E75", "grey": "#B4B2A9", "split": "#D85A30", "warn": "#BA7517"}
fig = plt.figure(figsize=(15, 9.5))
gs = gridspec.GridSpec(3, 2, height_ratios=[0.55, 2, 2], hspace=0.45, wspace=0.22)

tiles = [("Retained shares", f"{n_ret}", C["ok"]),
         ("Dividend payers", f"{payers}  ({payers / max(n_ret, 1) * 100:.0f}%)", C["div"]),
         ("Dividend events", f"{len(div_ev):,}", C["div"]),
         ("Split events", f"{len(spl_ev)}  ({spl_ev['Ticker'].nunique()} shares)", C["split"]),
         ("Unexplained jumps >35%", f"{n_jump}", C["warn"])]
top = gs[0, :].subgridspec(1, len(tiles), wspace=0.08)
for i, (lab, val, col) in enumerate(tiles):
    a = fig.add_subplot(top[0, i]); a.axis("off")
    a.add_patch(plt.Rectangle((0, 0), 1, 1, transform=a.transAxes, color=col, alpha=0.12))
    a.text(0.5, 0.60, val, ha="center", va="center", fontsize=17, fontweight="bold", color=col, transform=a.transAxes)
    a.text(0.5, 0.18, lab, ha="center", va="center", fontsize=10, transform=a.transAxes)

# dividend events per quarter
ax1 = fig.add_subplot(gs[1, 0])
if len(div_ev):
    q = div_ev.groupby(div_ev["Date"].dt.to_period("Q")).size()
    ax1.bar(q.index.astype(str), q.values, color=C["div"]); ax1.tick_params(axis="x", rotation=60)
else:
    ax1.text(0.5, 0.5, "No dividend events found", ha="center", va="center", transform=ax1.transAxes)
ax1.set_title("Dividend events per quarter"); ax1.set_ylabel("Events")

# top dividend payers (total dividends per share, USD)
ax2 = fig.add_subplot(gs[1, 1])
if len(div_ev):
    tp = div_ev.groupby("Ticker")["Dividends"].sum().nlargest(15).sort_values()
    ax2.barh(tp.index, tp.values, color=C["div"])
else:
    ax2.text(0.5, 0.5, "No dividend events found", ha="center", va="center", transform=ax2.transAxes)
ax2.set_title("Top 15 payers: total dividends per share (USD)")

# splits: count per quarter (when) + the 10 largest ratios (how big), instead of labelling every split
sub = gs[2, 0].subgridspec(1, 2, wspace=0.6, width_ratios=[1.15, 1])
ax3a, ax3b = fig.add_subplot(sub[0, 0]), fig.add_subplot(sub[0, 1])
if len(spl_ev):
    sq = spl_ev.groupby(spl_ev["Date"].dt.to_period("Q")).size()
    sq = sq.reindex(pd.period_range(START_TS, END_TS - pd.Timedelta(days=1), freq="Q"), fill_value=0)
    ax3a.bar(sq.index.astype(str), sq.values, color=C["split"]); ax3a.tick_params(axis="x", rotation=90, labelsize=7)
    ax3a.set_title("Split events per quarter"); ax3a.set_ylabel("Events")
    big = spl_ev.nlargest(10, "Stock Splits").sort_values("Stock Splits")
    ax3b.barh([f"{t} ({d:%b %Y})" for t, d in zip(big["Ticker"], big["Date"])], big["Stock Splits"], color=C["split"])
    for i_, v_ in enumerate(big["Stock Splits"]):
        ax3b.text(v_, i_, f" {v_:g}:1", va="center", fontsize=8)
    ax3b.set_xlim(0, big["Stock Splits"].max() * 1.3); ax3b.tick_params(axis="y", labelsize=8)
    ax3b.set_title("10 largest splits (new : 1)")
else:
    for a_ in (ax3a, ax3b):
        a_.text(0.5, 0.5, "No split events in the retained shares", ha="center", va="center", transform=a_.transAxes)

# payers vs non-payers
ax4 = fig.add_subplot(gs[2, 1])
ax4.pie([payers, max(n_ret - payers, 0)], labels=[f"Paid a dividend\n({payers})", f"No dividend\n({n_ret - payers})"],
        colors=[C["div"], C["grey"]], startangle=90, counterclock=False, wedgeprops=dict(width=0.38, edgecolor="white"))
ax4.set_title("Retained shares by dividend activity")

for a in (ax1, ax2, ax3a, ax3b):
    for s_ in ("top", "right"): a.spines[s_].set_visible(False)
fig.suptitle(f"Corporate actions in the retained shares, {start} to {end}", fontsize=15, y=0.995)
fig.text(0.5, 0.005, "Adj Close already reflects splits and dividends, so these shares are retained and flagged, not excluded. "
         "Yahoo Close and Volume are split-adjusted consistently.", ha="center", fontsize=9, color="#555")
plt.savefig(f"{OUT_DIR}/corporate_actions_infographic.png", dpi=200, bbox_inches="tight"); plt.show()

## 1.3 Weekly return and weekly beta

In [ ]:
prices_wide = (df_clean.pivot(index="Date", columns="Ticker", values="Adj Close").sort_index())

def calculate_weekly_returns(prices, shares=None):
    """prices: dates as rows, tickers as columns (Adj Close). Friday-ending weeks, last price in each week."""
    if shares is not None:
        prices = prices[shares]
    prices = prices.copy()
    prices.index = pd.to_datetime(prices.index)
    weekly = prices.sort_index().resample("W-FRI").last()

    # keep only complete calendar weeks inside the study period
    week_monday = weekly.index - pd.Timedelta(days=4)
    weekly = weekly[(week_monday >= START_TS) & (weekly.index < END_TS)]
    print("Complete weekly prices:", len(weekly), f"({weekly.index[0].date()} -> {weekly.index[-1].date()})")

    return weekly / weekly.shift(1) - 1        # NaN if the preceding week is missing - never filled

weekly_return = calculate_weekly_returns(prices_wide).iloc[1:]    # first week has no preceding week
print("Weekly returns:", weekly_return.shape)                     # expect 207 rows

expected_weeks = pd.date_range("2021-01-15", "2024-12-27", freq="W-FRI")
print("Missing weeks:", list(expected_weeks.difference(weekly_return.index)),
      "| extra weeks:", list(weekly_return.index.difference(expected_weeks)))

#### Mean weekly return

In [ ]:
market_ret = weekly_return[BENCH_SYMBOL]
share_ret  = weekly_return.drop(columns=BENCH_SYMBOL)

# only weeks where both the share and the benchmark have a return (common week labels)
aligned = share_ret.where(market_ret.notna(), axis=0)

mean_weekly_return = aligned.mean().rename("Mean_Weekly_Return")      # arithmetic mean, one value per share

print("Shares:", len(mean_weekly_return))
print("Weekly observations per share:", aligned.notna().sum().min(), "to", aligned.notna().sum().max())
print(mean_weekly_return.describe())
t = mean_weekly_return.index[0]
assert np.isclose(mean_weekly_return[t], weekly_return[t].dropna().mean())
mean_weekly_return.sort_values(ascending=False).head(10)

#### Weekly beta

In [ ]:
def calculate_weekly_beta(weekly_return, market_return, min_weeks=52):
    if min_weeks < 2:
        raise ValueError("min_weeks must be at least 2.")
    results = []
    for share in weekly_return.columns:
        paired = pd.concat([weekly_return[share].rename("share"), market_return.rename("market")], axis=1
                           ).replace([np.inf, -np.inf], np.nan).dropna()
        variance = paired["market"].var(ddof=1)
        beta = np.nan
        if len(paired) >= min_weeks and variance > 0:
            beta = paired["share"].cov(paired["market"]) / variance
        results.append({"Share": share, "Beta": beta, "Paired_weeks": len(paired)})
    return pd.DataFrame(results, columns=["Share", "Beta", "Paired_weeks"]).set_index("Share")

beta_df = calculate_weekly_beta(share_ret, market_ret)
print(beta_df["Paired_weeks"].value_counts())                         # expect 207 for every share

# sanity: the benchmark against itself must give beta = 1
print("Benchmark vs itself:", calculate_weekly_beta(weekly_return[[BENCH_SYMBOL]], market_ret).loc[BENCH_SYMBOL, "Beta"])
beta_df.describe()

## 1.4 Adapted RFM (daily data)

In [ ]:
RFM_WIN, BASE_WIN, MULT = 60, 20, 2.0

vol = (df_clean[df_clean["Ticker"] != BENCH_SYMBOL].pivot(index="Date", columns="Ticker", values="Volume").sort_index())
vol = vol.loc[vol.index < END_TS]

# observation date = final COMMON completed trading date (derived from the data, not typed in)
OBS_DATE = vol.dropna(how="any").index.max()
vol = vol.loc[:OBS_DATE]
print("Observation date:", OBS_DATE.date())

# baseline = mean of the 20 PREVIOUS sessions (shift(1) excludes the day being tested)
baseline = vol.shift(1).rolling(BASE_WIN).mean()
# high-volume day: strictly greater than 2 x baseline (first 20 sessions have no baseline)
high = (vol > MULT * baseline) & baseline.notna()

def last_event(col):
    days = col.index[col]
    return days[-1] if len(days) else pd.NaT

last_event_date = high.apply(last_event)
recency = (OBS_DATE - last_event_date).dt.days.rename("Recency")      # NaN if no event, never 0

rfm = pd.concat([last_event_date.rename("Last_Event_Date"), recency], axis=1)
rfm.index.name = "Ticker"

no_event = rfm.index[rfm["Recency"].isna()]
print("Shares with no event (Recency missing, excluded from RFM clustering):", len(no_event), list(no_event[:10]))
print(rfm["Recency"].value_counts().head())
rfm.head()

### Frequency

In [ ]:
window = high.loc[:OBS_DATE].tail(RFM_WIN)                  # 'high' was built on the full history
assert len(window) == RFM_WIN and window.index[-1] == OBS_DATE
print(f"Window: {window.index[0].date()} -> {window.index[-1].date()} ({len(window)} sessions)")

frequency = window.sum().astype(int).rename("Frequency")
rfm["Frequency"] = frequency
print(frequency.describe())
print(frequency.value_counts().sort_index().to_string())

### Monetary

In [ ]:
close = (df_clean[df_clean["Ticker"] != BENCH_SYMBOL].pivot(index="Date", columns="Ticker", values="Close")
         .sort_index().loc[:OBS_DATE])
close = close[vol.columns]

traded_value = close * vol                                   # daily Close x Volume
window_tv = traded_value.tail(RFM_WIN)
assert len(window_tv) == RFM_WIN and window_tv.index[-1] == OBS_DATE and window_tv.index.equals(window.index)

monetary = window_tv.sum().rename("Monetary")                # USD, approximate traded value
rfm["Monetary"] = monetary
print(monetary.describe().apply(lambda x: f"{x:,.0f}"))
print(monetary.sort_values(ascending=False).head(5).apply(lambda x: f"${x:,.0f}"))
rfm

### 1.5 Evidence: unit tests, alignment checks and worked example

In [ ]:
def volume_event_frame(volume, close_):
    d = pd.DataFrame({"Close": close_, "Volume": volume})
    d["Baseline20"]  = d["Volume"].shift(1).rolling(BASE_WIN).mean()
    d["Threshold"]   = MULT * d["Baseline20"]
    d["HighVolume"]  = (d["Volume"] > d["Threshold"]) & d["Baseline20"].notna()
    d["TradedValue"] = d["Close"] * d["Volume"]
    return d

# ---- unit tests of the rules (brief's own examples) ----
idx = pd.bdate_range("2024-01-01", periods=90)
v = pd.Series(1_000_000.0, index=idx); c = pd.Series(100.0, index=idx)
v.iloc[-1] = 2_500_000          # qualifies (threshold 2.0m)
v.iloc[-30] = 1_800_000         # does not qualify
t_ = volume_event_frame(v, c)
assert t_["Threshold"].iloc[-1] == 2_000_000 and t_["HighVolume"].iloc[-1] and not t_["HighVolume"].iloc[-30]
assert (idx[-1] - t_.index[t_["HighVolume"]][-1]).days == 0                      # event on obs date -> R = 0
assert (pd.Timestamp("2024-09-20") - pd.Timestamp("2024-09-15")).days == 5       # brief's recency example
assert 100 * 2_000_000 == 200_000_000                                            # brief's monetary example
flat = volume_event_frame(pd.Series(1_000_000.0, index=idx), c)
assert not flat["HighVolume"].any()                                              # no event -> Recency must be NaN, not 0
print("Unit tests passed")

# ---- alignment checks on the real data ----
s0 = vol.columns[0]
d0 = volume_event_frame(vol[s0], close[s0])
manual = vol[s0].iloc[-21:-1].mean()
assert np.isclose(d0["Baseline20"].iloc[-1], manual)                             # baseline = 20 PRECEDING sessions
v_mod = vol[s0].copy(); v_mod.iloc[-1] *= 1000
assert np.isclose(volume_event_frame(v_mod, close[s0])["Baseline20"].iloc[-1], manual)   # tested day excluded from its own baseline
assert (high[s0] == d0["HighVolume"]).all()                                      # helper agrees with vectorised version
assert vol.shape[0] >= BASE_WIN + RFM_WIN                                        # enough history for 20 + 60 sessions
print(f"Alignment checks passed | first testable day {vol.index[BASE_WIN].date()} | no-event shares: {len(no_event)}")

# ---- worked example for one share ----
EXAMPLE = "AAPL" if "AAPL" in vol.columns else vol.columns[0]
ex = volume_event_frame(vol[EXAMPLE], close[EXAMPLE])
win = ex.tail(RFM_WIN); events = ex.index[ex["HighVolume"]]
print(f"WORKED EXAMPLE: {EXAMPLE} | observation date {OBS_DATE.date()} | window {win.index[0].date()} -> {win.index[-1].date()}")
print(f"Baseline on obs date = mean of {ex.index[-21].date()}..{ex.index[-2].date()} = {ex['Baseline20'].iloc[-1]:,.0f}; threshold = {ex['Threshold'].iloc[-1]:,.0f}")
if len(events):
    print(f"RECENCY  : last high-volume day {events[-1].date()} -> {(OBS_DATE - events[-1]).days} calendar days")
print(f"FREQUENCY: {int(win['HighVolume'].sum())} days -> {[str(x.date()) for x in win.index[win['HighVolume']]]}")
print(f"MONETARY : ${win['TradedValue'].sum():,.0f}")
assert (rfm.loc[EXAMPLE, "Frequency"], round(rfm.loc[EXAMPLE, "Monetary"])) == (int(win["HighVolume"].sum()), round(win["TradedValue"].sum()))
print(win.loc[win["HighVolume"], ["Close", "Volume", "Baseline20", "Threshold", "TradedValue"]].round(2).to_string())
win.to_csv(f"{OUT_DIR}/worked_example_{EXAMPLE}.csv")

## 2. Feature matrices, clustering and sensitivity

In [ ]:
# one seed and one n_init for EVERY K-means fit in this notebook
SEED, N_INIT, MIN_FRAC = 42, 50, 0.05
np.random.seed(SEED)

# weekly features + beta in one long table (Week, Share, Weekly_Return, then per-share values)
base = (weekly_return.drop(columns=BENCH_SYMBOL).rename_axis("Week").reset_index()
        .melt(id_vars="Week", var_name="Share", value_name="Weekly_Return")
        .sort_values(["Share", "Week"]).reset_index(drop=True))
mwr = mean_weekly_return.rename("Mean_Weekly_Return").rename_axis("Share").reset_index()
bt  = beta_df[["Beta"]].rename_axis("Share").reset_index()
combined = base.merge(mwr, on="Share", how="left").merge(bt, on="Share", how="left")
print(combined.shape)
combined.head()

In [ ]:
plot_df = rfm[["Recency", "Frequency", "Monetary"]].dropna().copy()
plot_df["Log10_Monetary"] = np.log10(plot_df["Monetary"])

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a, (col, label, colour) in zip(ax, [("Recency", "Recency (calendar days)", "#1D9E75"),
                                        ("Frequency", "Frequency (spike days in last 60)", "#534AB7"),
                                        ("Log10_Monetary", "log10 Monetary (USD)", "#BA7517")]):
    a.hist(plot_df[col], bins=30, color=colour, edgecolor="white")
    a.axvline(plot_df[col].median(), color="black", ls="--", lw=1)
    a.set_title(label, fontsize=11); a.set_ylabel("Number of shares")
    a.text(0.97, 0.93, f"median = {plot_df[col].median():.1f}", transform=a.transAxes, ha="right", va="top", fontsize=9)
    for s_ in ("top", "right"): a.spines[s_].set_visible(False)
fig.suptitle(f"Adapted RFM distributions | observation date {OBS_DATE.date()} | n = {len(plot_df)} shares", fontsize=13)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/rfm_distributions.png", dpi=200); plt.show()

### 2.1 Feature preparation and four baseline solutions

In [ ]:
# ! pip install -U threadpoolctl scikit-learn      # only if KMeans raises the threadpool error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

# ---------- 1. common valid share set (all five features present) ----------
all_feat = pd.concat([mean_weekly_return.rename("Mean_Weekly_Return").rename_axis("Share"),
                      beta_df["Beta"].rename_axis("Share"),
                      rfm[["Recency", "Frequency", "Monetary"]].rename_axis("Share")],
                     axis=1, join="inner").dropna()
print("Shares in common valid set:", len(all_feat))

# ---------- final inclusion / exclusion log ----------
final_log = check[["Reason", "Status"]].copy()
final_log["Final_Status"] = final_log["Status"]; final_log["Final_Reason"] = final_log["Reason"]
for tk in final_log.index[final_log["Status"] == "Included"]:
    if tk not in all_feat.index:
        why = "no high-volume event (Recency missing)" if tk in no_event else "missing weekly feature or fewer than 52 paired weeks"
        final_log.loc[tk, ["Final_Status", "Final_Reason"]] = ["Excluded from clustering", why]
final_log.loc[final_log["Final_Status"] == "Included", "Final_Status"] = "Included (common valid set)"
final_log.to_csv(f"{OUT_DIR}/final_inclusion_exclusion_log.csv")
print(final_log["Final_Status"].value_counts().to_string())

# ---------- 2. inspect skew and outliers ----------
def inspect(df):
    q1, q3 = df.quantile(.25), df.quantile(.75); iqr = q3 - q1
    out = ((df < q1 - 1.5 * iqr) | (df > q3 + 1.5 * iqr)).sum()
    return pd.DataFrame({"mean": df.mean(), "median": df.median(), "std": df.std(), "min": df.min(), "max": df.max(),
                         "skew": df.skew(), "IQR_outliers": out, "outlier_%": (out / len(df) * 100).round(1)})

print("RAW FEATURES"); print(inspect(all_feat).round(3).to_string())

fig, ax = plt.subplots(2, 5, figsize=(20, 6))
for i, c_ in enumerate(all_feat.columns):
    ax[0, i].hist(all_feat[c_], bins=30, edgecolor="white"); ax[0, i].set_title(c_)
    ax[1, i].boxplot(all_feat[c_], vert=False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/feature_skew_outliers.png", dpi=150); plt.show()

# ---------- 3. two separate matrices; log1p on Monetary only ----------
risk_return = all_feat[["Mean_Weekly_Return", "Beta"]].copy()
activity    = all_feat[["Recency", "Frequency", "Monetary"]].copy()
activity["Monetary"] = np.log1p(activity["Monetary"])
activity = activity.rename(columns={"Monetary": "log1p_Monetary"})
print("\nSkew of Monetary before / after log1p:", round(all_feat["Monetary"].skew(), 2), "->", round(activity["log1p_Monetary"].skew(), 2))
print("\nACTIVITY MATRIX AFTER log1p"); print(inspect(activity).round(3).to_string())

# ---------- 4. standardise each matrix separately ----------
def standardise(df):
    return pd.DataFrame(StandardScaler().fit_transform(df), index=df.index, columns=df.columns)
X_risk, X_act = standardise(risk_return), standardise(activity)
print("\nStandardised means ~0, std ~1:"); print(X_risk.agg(["mean", "std"]).round(3)); print(X_act.agg(["mean", "std"]).round(3))

# ---------- 5. four baseline solutions ----------
def fit(method, X, k, seed=SEED):
    if method == "KMeans":
        return KMeans(n_clusters=k, n_init=N_INIT, random_state=seed, algorithm="elkan").fit_predict(X)
    return AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X)

matrices = {"Risk_Return": X_risk, "Activity": X_act}
K = 3
summary = []
baseline_labels = pd.DataFrame(index=all_feat.index)
for name, X in matrices.items():
    for method in ["KMeans", "Agglomerative"]:
        lab = fit(method, X, K)
        baseline_labels[f"{name}_{method}"] = lab
        summary.append({"Matrix": name, "Method": method, "k": K, "Silhouette": silhouette_score(X, lab), "Sizes": np.bincount(lab).tolist()})
print("\nFOUR BASELINE SOLUTIONS"); print(pd.DataFrame(summary).round(3).to_string(index=False))
all_feat.join(baseline_labels).to_csv(f"{OUT_DIR}/feature_table_baseline_assignments_k3.csv")   # one row per included share

#### Checks after log1p and standardisation (2.1)

In [ ]:
# checks_after_transform
from scipy import stats

# ---- 1. integrity: same shares, no NaN or infinite values in either final matrix ----
for name, X in matrices.items():
    assert X.shape[0] == len(all_feat) and np.isfinite(X.values).all(), f"problem in {name}"
print(f"Both matrices: {len(all_feat)} shares, no missing or infinite values")

# ---- 2. scaling must not change relationships: correlations before and after standardising ----
assert np.allclose(risk_return.corr().values, X_risk.corr().values) and np.allclose(activity.corr().values, X_act.corr().values)
print("Correlations are identical before and after standardising (scaling only shifts and rescales)")

# ---- 3. shape and outliers in STANDARDISED units ----
rows = []
for name, X in matrices.items():
    for c_ in X.columns:
        z = X[c_]
        rows.append({"Matrix": name, "Feature": c_, "mean": z.mean(), "std": z.std(ddof=0), "skew": z.skew(), "kurtosis": z.kurt(),
                     "min": z.min(), "max": z.max(), "|z|>3": int((z.abs() > 3).sum()), "|z|>4": int((z.abs() > 4).sum())})
zchk = pd.DataFrame(rows); print(); print(zchk.round(2).to_string(index=False))
zchk.to_csv(f"{OUT_DIR}/standardised_feature_checks.csv", index=False)
print("\nNote: standardising does not change skew. Skew is reduced only by log1p, so Recency stays skewed and its outliers remain.")

# ---- 4. Monetary: raw vs log1p vs standardised, with Q-Q plots against a normal distribution ----
raw_m = all_feat["Monetary"]; log_m = activity["log1p_Monetary"]; std_m = X_act["log1p_Monetary"]
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for j, (v, t_) in enumerate([(raw_m / 1e9, "Monetary, raw (USD bn)"), (log_m, "log1p(Monetary)"), (std_m, "log1p(Monetary), standardised")]):
    ax[0, j].hist(v, bins=40, color="#BA7517", edgecolor="white"); ax[0, j].set_title(f"{t_}\nskew = {stats.skew(v):.2f}")
    stats.probplot(v, dist="norm", plot=ax[1, j]); ax[1, j].set_title("Q-Q plot vs normal")
    for a_ in (ax[0, j], ax[1, j]): a_.spines["top"].set_visible(False); a_.spines["right"].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/monetary_before_after.png", dpi=200); plt.show()

# ---- 5. all standardised features on one scale: box plots with +/-3 reference lines ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for a_, (name, X) in zip(ax, matrices.items()):
    a_.boxplot([X[c_] for c_ in X.columns]); a_.set_xticklabels(X.columns)
    for y_ in (-3, 3): a_.axhline(y_, color="red", ls="--", lw=0.8)
    a_.set_title(f"{name}: standardised features (red = +/-3 sd)"); a_.set_ylabel("z-score")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/standardised_boxplots.png", dpi=200); plt.show()

## 2.2 Agglomerative clustering (Ward linkage, Euclidean distance)

**Suitability.** Ward merges the pair of clusters that least increases within-cluster variance, which needs Euclidean distance on standardised features. It suits continuous features and a sample of about 500 shares, and the dendrogram shows the hierarchy without fixing k in advance.
**Limitations.** Outliers can form tiny late-merging clusters (Recency is the main concern); ties at Recency = 11 add little information; Ward favours compact, similar-sized clusters; merges cannot be undone; the cut line alone does not decide k.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, cophenet, fcluster
from scipy.spatial.distance import pdist

COLS = {"Risk_Return": ["Mean_Weekly_Return", "Beta"], "Activity": ["Recency", "Frequency", "Monetary"]}
def profile(labels, cols):
    """cluster medians in ORIGINAL units, with sizes"""
    g = all_feat.assign(Cluster=labels).groupby("Cluster")
    out = g[cols].median().add_suffix("_median")
    out["n"] = g.size(); out["%"] = (out["n"] / len(all_feat) * 100).round(1)
    return out.round(4)

# ---------- Ward: compare k = 2..8 on silhouette, cluster sizes and feasibility ----------
rows = []
for name, X in matrices.items():
    for k in range(2, 9):
        lab = fit("Agglomerative", X, k); s_ = np.bincount(lab)
        rows.append({"Matrix": name, "Method": "Agglomerative", "k": k, "Inertia": np.nan,
                     "Silhouette": round(silhouette_score(X, lab), 3),
                     "Min_%": round(s_.min() / len(X) * 100, 1), "Sizes": sorted(s_.tolist(), reverse=True)})
ward_comp = pd.DataFrame(rows)
ward_comp["Feasible"] = ward_comp["Min_%"] >= MIN_FRAC * 100          # smallest cluster at least 5% of shares
print(ward_comp.drop(columns="Inertia").to_string(index=False))
ward_comp.to_csv(f"{OUT_DIR}/ward_k_comparison.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a_, name in zip(ax, matrices):
    d_ = ward_comp[ward_comp.Matrix == name]
    a_.plot(d_["k"], d_["Silhouette"], "s--", color="#1D9E75")
    bad = d_[~d_["Feasible"]]; a_.scatter(bad["k"], bad["Silhouette"], s=120, facecolors="none", edgecolors="red")
    a_.set_title(f"{name}: Ward silhouette (red ring = smallest cluster below 5%)"); a_.set_xlabel("k"); a_.set_xticks(range(2, 9))
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/ward_k_selection.png", dpi=200); plt.show()

# ---------- Ward: interpretable profiles of the candidate k (original units) ----------
for name, X in matrices.items():
    for k in [3, 4, 5]:
        print(f"\n=== {name}, Ward, k = {k} ==="); print(profile(fit("Agglomerative", X, k), COLS[name]).to_string())

In [ ]:
# ---------- SET THE WARD k after reading the table, the profiles and the dendrogram ----------
SELECTED_WARD = {"Risk_Return": 3, "Activity": 4}

# ---------- one Ward dendrogram per matrix, cut at the selected k (supporting evidence only) ----------
for name, X in matrices.items():
    k = SELECTED_WARD[name]
    Z = linkage(X.values, method="ward", metric="euclidean")
    cut = (Z[-k, 2] + Z[-(k - 1), 2]) / 2
    assert len(set(fcluster(Z, k, "maxclust"))) == k
    fig, ax = plt.subplots(figsize=(12, 4.5))
    dendrogram(Z, truncate_mode="lastp", p=30, no_labels=True, color_threshold=cut, ax=ax)
    ax.axhline(cut, color="black", ls="--", lw=1)
    ax.set_title(f"{name}: Ward, Euclidean, standardised | cut at k = {k}"); ax.set_ylabel("Ward merge distance")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/dendrogram_{name}.png", dpi=200); plt.show()
    c_, _ = cophenet(Z, pdist(X.values))
    print(f"{name}: cophenetic r = {c_:.3f} | last 8 merge heights = {np.round(Z[-8:, 2], 1)}")
    print(f"=== {name}, Ward, selected k = {k} ==="); print(profile(fit("Agglomerative", X, k), COLS[name]).to_string())

## 2.3 K-means clustering

**Suitability.** K-means is fast and gives interpretable centres. It works well on standardised, roughly symmetric features, and is reproducible once the seed and initialisations are fixed.
**Limitations.** k must be chosen in advance and the criteria can disagree; it assumes round clusters of similar size; means are pulled by outliers; ties at Recency = 11 stack many shares at one point; different starts can end in different local minima, hence the fixed seed and multiple initialisations.

In [ ]:
# ---------- K-means: compare k = 2..8 on silhouette, inertia and cluster sizes (seed and n_init fixed above) ----------
rows = []
for name, X in matrices.items():
    for k in range(2, 9):
        km = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED, algorithm="elkan").fit(X); s_ = np.bincount(km.labels_)
        rows.append({"Matrix": name, "Method": "KMeans", "k": k, "Inertia": km.inertia_,
                     "Silhouette": round(silhouette_score(X, km.labels_), 3),
                     "Min_%": round(s_.min() / len(X) * 100, 1), "Sizes": sorted(s_.tolist(), reverse=True)})
km_comp = pd.DataFrame(rows)
km_comp["Inertia_drop_%"] = (km_comp.groupby("Matrix")["Inertia"].pct_change() * -100).round(1)   # gain from adding a cluster
km_comp["Feasible"] = km_comp["Min_%"] >= MIN_FRAC * 100
print(km_comp.round(3).to_string(index=False))
km_comp.to_csv(f"{OUT_DIR}/kmeans_k_comparison.csv", index=False)

# one combined table (Ward + K-means) used later in the notebook
comp = pd.concat([ward_comp, km_comp], ignore_index=True)
comp.to_csv(f"{OUT_DIR}/cluster_count_comparison.csv", index=False)

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
for j, name in enumerate(matrices):
    d_ = km_comp[km_comp.Matrix == name]
    ax[0, j].plot(d_["k"], d_["Inertia"], "o-"); ax[0, j].set_title(f"{name}: inertia (elbow)")
    ax[1, j].plot(d_["k"], d_["Silhouette"], "o-", color="#1D9E75"); ax[1, j].set_title(f"{name}: silhouette")
    bad = d_[~d_["Feasible"]]; ax[1, j].scatter(bad["k"], bad["Silhouette"], s=120, facecolors="none", edgecolors="red")
    for a_ in ax[:, j]: a_.set_xticks(range(2, 9)); a_.set_xlabel("k")
plt.suptitle(f"K-means, seed {SEED}, {N_INIT} initialisations | red ring = smallest cluster below 5%")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/kmeans_k_selection.png", dpi=200); plt.show()

# ---------- K-means: interpretable profiles of the candidate k (original units) ----------
for name, X in matrices.items():
    for k in [3, 4, 5]:
        print(f"\n=== {name}, K-means, k = {k} ==="); print(profile(fit("KMeans", X, k), COLS[name]).to_string())

In [ ]:
# ---------- SET THE K-MEANS k after reading the table, the elbow and the profiles ----------
SELECTED_KM = {"Risk_Return": 3, "Activity": 4}

# ---------- reproducibility: seeds and initialisations ----------
for name, X in matrices.items():
    k = SELECTED_KM[name]
    base_lab = fit("KMeans", X, k, seed=SEED)
    aris = [adjusted_rand_score(base_lab, fit("KMeans", X, k, seed=s_)) for s_ in [1, 7, 99, 2024]]
    print(f"{name}, k = {k}: agreement with 4 other seeds (ARI) = {np.round(aris, 3)}")
    print(f"   inertia n_init=1: {KMeans(k, n_init=1, random_state=3, algorithm='elkan').fit(X).inertia_:.2f}"
          f" vs n_init={N_INIT}: {KMeans(k, n_init=N_INIT, random_state=SEED, algorithm='elkan').fit(X).inertia_:.2f}")
    print(f"=== {name}, K-means, selected k = {k} ==="); print(profile(base_lab, COLS[name]).to_string())

# ---------- final assignments for both algorithms (same shares) ----------
SELECTED = SELECTED_WARD                     # 2.5.2 repeats the selected WARD solution
final_labels = pd.DataFrame(index=all_feat.index)
for name, X in matrices.items():
    final_labels[f"{name}_KMeans"]        = fit("KMeans", X, SELECTED_KM[name])
    final_labels[f"{name}_Agglomerative"] = fit("Agglomerative", X, SELECTED_WARD[name])
final_labels.to_csv(f"{OUT_DIR}/final_cluster_labels.csv")

### 2.4 Cluster profiles and algorithm comparison (same shares, selected k)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score
from scipy.optimize import linear_sum_assignment

# ---- profiles in ORIGINAL units for both algorithms ----
for name in matrices:
    for method in ["KMeans", "Agglomerative"]:
        kk = SELECTED_KM[name] if method == "KMeans" else SELECTED_WARD[name]
        print(f"\n=== {name} | {method} | k = {kk} (medians, original units) ===")
        print(profile(final_labels[f"{name}_{method}"].values, COLS[name]).to_string())

# ---- labelled 2D visualisation ----
# Risk/return has only two features, so it is plotted directly (no reduction).
# Activity has three, so PCA projects the standardised data to 2D; axes show the variance each component keeps.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for r_, (name, X) in enumerate(matrices.items()):
    if name == "Risk_Return":
        xy = X.values; xl, yl = "Mean weekly return (standardised)", "Beta (standardised)"
    else:
        p2 = PCA(n_components=2).fit(X); xy = p2.transform(X); ev = p2.explained_variance_ratio_
        xl, yl = f"PC1 ({ev[0] * 100:.0f}% of variance)", f"PC2 ({ev[1] * 100:.0f}% of variance)"
        print("\nPCA loadings (activity):"); print(pd.DataFrame(p2.components_.T, index=X.columns, columns=["PC1", "PC2"]).round(3))
        print(f"2D projection keeps {ev.sum() * 100:.0f}% of the variance; distances in the plot are approximate.")
    for c_, method in enumerate(["KMeans", "Agglomerative"]):
        lab = final_labels[f"{name}_{method}"].values; ax = axes[r_, c_]
        ax.scatter(xy[:, 0], xy[:, 1], c=lab, cmap="tab10", s=18, alpha=0.75)
        for k_ in np.unique(lab):
            m_ = lab == k_
            ax.annotate(f"C{k_} (n={m_.sum()})", xy[m_].mean(axis=0), ha="center", fontsize=10, fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8))
        ax.set_title(f"{name}: {method}, k = {(SELECTED_KM if method == 'KMeans' else SELECTED_WARD)[name]}"); ax.set_xlabel(xl); ax.set_ylabel(yl)
plt.suptitle("Clusters on the same shares (cluster numbers are arbitrary and differ between algorithms)", y=0.995)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/cluster_scatter_2d.png", dpi=200); plt.show()

# ---- algorithm comparison on the SAME shares ----
rows = []
for name, X in matrices.items():
    a_, b_ = final_labels[f"{name}_KMeans"], final_labels[f"{name}_Agglomerative"]
    ct = pd.crosstab(a_, b_, rownames=["KMeans"], colnames=["Ward"])
    ri, ci = linear_sum_assignment(-ct.values)                 # best label matching (labels are arbitrary)
    print(f"\n{name}: cross-tabulation, shares = {len(a_)}"); print(ct)
    rows.append({"Matrix": name, "k_KMeans": SELECTED_KM[name], "k_Ward": SELECTED_WARD[name], "Shares": len(a_),
                 "ARI": round(adjusted_rand_score(a_, b_), 3),
                 "Matched_agreement_%": round(ct.values[ri, ci].sum() / ct.values.sum() * 100, 1),
                 "Silhouette_KMeans": round(silhouette_score(X, a_), 3), "Silhouette_Ward": round(silhouette_score(X, b_), 3),
                 "DaviesBouldin_KMeans": round(davies_bouldin_score(X, a_), 3), "DaviesBouldin_Ward": round(davies_bouldin_score(X, b_), 3)})
algo_cmp = pd.DataFrame(rows); print(); print(algo_cmp.to_string(index=False))
algo_cmp.to_csv(f"{OUT_DIR}/algorithm_comparison.csv", index=False)

### 2.5.1 Correlation analysis among R, F and log1p(M)

In [ ]:
from sklearn.decomposition import PCA

act = all_feat[["Recency", "Frequency", "Monetary"]].copy()
act["log1p_Monetary"] = np.log1p(act["Monetary"])
act = act[["Recency", "Frequency", "log1p_Monetary"]]

pearson, spearman = act.corr(method="pearson"), act.corr(method="spearman")
print("PEARSON\n", pearson.round(3), "\n\nSPEARMAN\n", spearman.round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for a, c_, t_ in zip(ax, [pearson, spearman], ["Pearson", "Spearman"]):
    sns.heatmap(c_, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=a, cbar=False); a.set_title(f"{t_}: R, F, log1p(M)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/rfm_correlations.png", dpi=200); plt.show()

# shared information after standardisation
Zs = (act - act.mean()) / act.std()
pca = PCA().fit(Zs)
print("\nVariance share by principal component:", np.round(pca.explained_variance_ratio_, 3))
print("PC1 loadings:\n", pd.Series(pca.components_[0], index=act.columns).round(3))

# VIF = diagonal of the inverse correlation matrix (above about 5 suggests strong overlap)
vif = pd.Series(np.diag(np.linalg.inv(act.corr().values)), index=act.columns)
print("\nVIF:", vif.round(2).to_dict())

# do the activity features overlap with risk and return?
cross = all_feat[["Mean_Weekly_Return", "Beta"]].join(act).corr().loc[["Mean_Weekly_Return", "Beta"], act.columns]
print("\nRisk/return vs activity correlations:\n", cross.round(3))

### 2.5.2 Removal of Monetary (Ward, R and F only)

In [ ]:
K_ACT = SELECTED_WARD["Activity"]
idx = X_act.index                                           # same shares as the full RFM solution

lab_full = fit("Agglomerative", X_act, K_ACT)               # R, F, log1p(M)
rf = all_feat.loc[idx, ["Recency", "Frequency"]]
X_rf = pd.DataFrame(StandardScaler().fit_transform(rf), index=idx, columns=rf.columns)   # standardised afresh
lab_rf = fit("Agglomerative", X_rf, K_ACT)                  # same k

ari = adjusted_rand_score(lab_full, lab_rf)
print(f"Shares: {len(idx)} | k = {K_ACT} | ARI (R,F,M vs R,F only) = {ari:.3f}")
print(f"Silhouette: full = {silhouette_score(X_act, lab_full):.3f} | R,F only = {silhouette_score(X_rf, lab_rf):.3f}")
print(pd.crosstab(pd.Series(lab_full, index=idx, name="Full_RFM"), pd.Series(lab_rf, index=idx, name="RF_only")))

d = all_feat.loc[idx, ["Recency", "Frequency", "Monetary"]].copy(); d["log1p_M"] = np.log1p(d["Monetary"])
def tv_profile(labels, title):
    g = d.assign(Cluster=labels).groupby("Cluster")
    out = pd.DataFrame({"n": g.size(), "R_median": g["Recency"].median(), "F_median": g["Frequency"].median(),
                        "M_median_USD_bn": (g["Monetary"].median() / 1e9).round(2),
                        "M_IQR_USD_bn": ((g["Monetary"].quantile(.75) - g["Monetary"].quantile(.25)) / 1e9).round(2)})
    print(f"\n=== {title} ==="); print(out.round(2).to_string())
tv_profile(lab_full, "Full solution: R, F, log1p(M)")
tv_profile(lab_rf, "Reduced solution: R, F only")

def eta_sq(labels, col):                                    # share of variance in a feature explained by the clusters
    s_ = d[col]; tot = ((s_ - s_.mean()) ** 2).sum()
    return sum(len(g) * (g.mean() - s_.mean()) ** 2 for _, g in s_.groupby(labels)) / tot
for col in ["log1p_M", "Recency", "Frequency"]:
    print(f"{col:8s} variance explained: full = {eta_sq(lab_full, col):.2f} | R,F only = {eta_sq(lab_rf, col):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, labs, t_ in zip(ax, [lab_full, lab_rf], ["Full: R, F, log1p(M)", "Reduced: R, F only"]):
    data = [np.log10(d.loc[labs == c_, "Monetary"]) for c_ in range(K_ACT)]
    a.boxplot(data); a.set_xticklabels([f"C{c_} (n={len(x)})" for c_, x in enumerate(data)]); a.set_title(t_); a.set_xlabel("Cluster")
ax[0].set_ylabel("log10 traded value, 60 sessions (USD)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/removal_of_monetary.png", dpi=200); plt.show()
# Traded value is only a liquidity proxy: spread and price impact are not measured here.

### Return dependencies between clusters (evidence for 3.2.3)

In [ ]:
# Beta differences alone do not establish diversification: shares in different clusters can still move together.
ret_c = share_ret[all_feat.index]                                   # weeks x shares (the 486 clustered shares)
lab_rr = final_labels["Risk_Return_KMeans"]
cluster_ret = ret_c.T.groupby(lab_rr.values).mean().T               # average weekly return of each risk/return cluster
cl_corr = cluster_ret.corr()
print("Correlation of cluster-average weekly returns:"); print(cl_corr.round(2))

pair = ret_c.corr().values; off = ~np.eye(len(pair), dtype=bool)
print(f"\nAverage pairwise correlation of ALL shares: {pair[off].mean():.2f}")
same = lab_rr.values[:, None] == lab_rr.values[None, :]
print(f"  within the same cluster : {pair[off & same].mean():.2f}")
print(f"  across different clusters: {pair[off & ~same].mean():.2f}")
print(f"Median R-squared of share vs benchmark: {np.median([ret_c[c_].corr(market_ret.loc[ret_c.index]) ** 2 for c_ in ret_c.columns]):.2f}")

plt.figure(figsize=(4.8, 3.8)); sns.heatmap(cl_corr, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False)
plt.title("Cluster-average weekly return correlations"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/cluster_return_correlations.png", dpi=200); plt.show()

# Task 3: Business Intelligence and Critical Review

Run every cell above first (this section reads `all_feat`, `final_labels`, `sp500_df` and the saved figures).
Replace each `[ ]` below with numbers from your own output before submitting. Word limits: 3.1 is 200 words, 3.2 is 500 words (visualisations excluded).

## 3.1 Data mart design

**Grain, facts and dimensions.** Grain: one row per share per trading day (fact_daily_market); derived metrics sit at share, observation date and run grain (fact_share_features). Daily facts: adjusted close, close, volume, traded value, high-volume flag, 20-session baseline. Derived facts: weekly return, beta, R, F, M, cluster label. Dimensions: date (Friday week-ending, trading-day flag), share, sector (GICS, retrieval date), benchmark (^GSPC, price basis).

**Provenance.** A source table records provider, retrieval timestamp, ticker-list date and download parameters. A run table stores study period, observation date, 20- and 60-session windows, 2x threshold, seed, initialisations, k, software versions and a calculation version. Feature and cluster tables carry run_id, so any result can be regenerated. Raw data are append-only; nothing is overwritten or filled.

**Support.** The star schema lets analysts join features, clusters and sectors without recomputation, slice by sector or cluster, and compare runs. Portfolio analysts obtain beta and return profiles, monitoring teams activity flags, and reviewers exclusion logs and audit trails. Outputs remain descriptive.

## 3.2 Review of results and visualisation

### Visualisations for 3.2.1 (profiles, sector composition, overlap)

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import adjusted_rand_score

METHOD = "KMeans"                                   # or "Agglomerative"; use the same method in the text
rr, ac = f"Risk_Return_{METHOD}", f"Activity_{METHOD}"

# ---------- sector lookup (current GICS sector from the Wikipedia table) ----------
sector = (sp500_df.assign(Share=sp500_df["Symbol"].str.replace(".", "-", regex=False))
          .set_index("Share")["GICS Sector"].rename("Sector"))
prof = all_feat.join(final_labels).join(sector)
print("Shares without a sector:", int(prof["Sector"].isna().sum()))

# ---------- 1. profiles in ORIGINAL units ----------
rr_prof = prof.groupby(rr)[["Mean_Weekly_Return", "Beta"]].median()
ac_prof = prof.groupby(ac)[["Recency", "Frequency", "Monetary"]].median()
ac_prof["Monetary"] = ac_prof["Monetary"] / 1e9                    # USD billions
ac_prof = ac_prof.rename(columns={"Monetary": "Monetary_USD_bn"})
for t_, p_, c_ in [("Risk and return", rr_prof, rr), ("Trading activity", ac_prof, ac)]:
    p_["n"] = prof.groupby(c_).size(); p_["%"] = (p_["n"] / len(prof) * 100).round(1)
    print(f"\n{t_} clusters ({METHOD}), medians in original units"); print(p_.round(4).to_string())

# ---------- Figure 1: profile heatmaps (colour = z-score of cluster median, numbers = original units) ----------
def prof_heat(ax, tbl, title):
    vals = tbl.drop(columns=["n", "%"]); z = (vals - vals.mean()) / vals.std(ddof=0)
    z.index = [f"C{i} (n={tbl.loc[i, 'n']})" for i in tbl.index]
    sns.heatmap(z, annot=vals.round(3).astype(str).values, fmt="", cmap="RdBu_r", center=0, cbar=False, ax=ax)
    ax.set_title(title); ax.set_ylabel("")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
prof_heat(ax[0], rr_prof, f"Risk and return clusters ({METHOD})"); prof_heat(ax[1], ac_prof, f"Trading activity clusters ({METHOD})")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig1_cluster_profiles.png", dpi=200); plt.show()

# ---------- Figure 2: sector composition of each cluster (100% stacked bars) ----------
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), sharex=True)
for a, col, t_ in zip(ax, [rr, ac], ["Risk and return clusters", "Trading activity clusters"]):
    ct = pd.crosstab(prof[col], prof["Sector"], normalize="index") * 100
    ct.index = [f"C{i} (n={(prof[col] == i).sum()})" for i in ct.index]
    ct.plot(kind="barh", stacked=True, ax=a, colormap="tab20", legend=False, width=0.8)
    a.set_title(f"Sector mix: {t_}"); a.set_xlabel("% of cluster")
h, l = ax[0].get_legend_handles_labels()
fig.legend(h, l, loc="center right", fontsize=8, title="GICS sector")
plt.tight_layout(rect=[0, 0, 0.84, 1]); plt.savefig(f"{OUT_DIR}/fig2_sector_composition.png", dpi=200); plt.show()

# ---------- Figure 3: over/under-representation vs the whole sample (lift; 1.0 = same as overall mix) ----------
overall = prof["Sector"].value_counts(normalize=True)
fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
for a, col, t_ in zip(ax, [rr, ac], ["Risk and return", "Trading activity"]):
    lift = pd.crosstab(prof[col], prof["Sector"], normalize="index").div(overall, axis=1)
    sns.heatmap(lift, annot=True, fmt=".1f", cmap="RdBu_r", center=1, cbar=False, ax=a)
    a.set_title(f"Sector lift: {t_} (1.0 = same as overall)"); a.set_ylabel("Cluster"); a.set_xlabel("")
    a.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_sector_lift.png", dpi=200); plt.show()

# ---------- Figure 4: do the two clusterings overlap? ----------
ct = pd.crosstab(prof[rr].rename("Risk/return cluster"), prof[ac].rename("Activity cluster"))
plt.figure(figsize=(5.5, 3.6)); sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Risk/return vs activity clusters (ARI = {adjusted_rand_score(prof[rr], prof[ac]):.2f})")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4_cluster_overlap.png", dpi=200); plt.show()

prof.to_csv(f"{OUT_DIR}/share_features_clusters_sectors.csv")
# Sensitivity figures already saved earlier: rfm_correlations.png and removal_of_monetary.png

### Visualisations for 3.2.2 (correlations and removal of Monetary, saved in section 2.5)

In [ ]:
from IPython.display import Image, display
for f in ["rfm_correlations.png", "removal_of_monetary.png"]:
    display(Image(filename=f"{OUT_DIR}/{f}"))

### Numbers for the written review

In [ ]:
# ---------- numbers for the [ ] gaps in the written review (everything printed comes from your own results) ----------
overall = prof["Sector"].value_counts(normalize=True) * 100
print("=== 3.2.1  cluster profiles, sector pattern and separation ===")
for col, mat in [(rr, "Risk_Return"), (ac, "Activity")]:
    k = (SELECTED_KM if METHOD == "KMeans" else SELECTED_WARD)[mat]
    sil = comp[(comp.Matrix == mat) & (comp.Method == METHOD) & (comp.k == k)]["Silhouette"].iloc[0]
    print(f"\n{mat} | {METHOD} | k = {k} | silhouette = {sil}")
    share = pd.crosstab(prof[col], prof["Sector"], normalize="index") * 100
    for c_ in share.index:
        top = share.loc[c_].sort_values(ascending=False).head(2)
        print(f"  C{c_} (n={(prof[col] == c_).sum()}): top sectors -> " +
              ", ".join(f"{s} {v:.0f}% (index {overall[s]:.0f}%)" for s, v in top.items()))
    lift = share.div(overall, axis=1)
    strong = [(c_, s, round(lift.loc[c_, s], 2)) for c_ in lift.index for s in lift.columns if lift.loc[c_, s] >= 1.5]
    print("  Sectors over-represented (lift >= 1.5):", strong if strong else "none")
print(f"\nOverlap of the two clusterings, adjusted Rand index = {adjusted_rand_score(prof[rr], prof[ac]):.2f}")

print("\n=== 3.2.2  correlations and Monetary removal ===")
print(f"Recency vs Frequency: Pearson {pearson.loc['Recency', 'Frequency']:.2f} | Spearman {spearman.loc['Recency', 'Frequency']:.2f}")
print(f"log1p Monetary vs Recency {pearson.loc['log1p_Monetary', 'Recency']:.2f} | vs Frequency {pearson.loc['log1p_Monetary', 'Frequency']:.2f}")
print(f"First principal component explains {pca.explained_variance_ratio_[0] * 100:.0f}% of variance in R, F, log1p(M)")
e_full, e_rf = eta_sq(lab_full, "log1p_M"), eta_sq(lab_rf, "log1p_M")
print(f"Monetary removal: ARI = {adjusted_rand_score(lab_full, lab_rf):.2f} | variance in log1p(M) explained by clusters: {e_full:.2f} -> {e_rf:.2f}")
print("Reading:", "Monetary mainly separates LIQUIDITY levels (clusters lose most of their traded-value differences without it)"
      if e_full - e_rf > 0.3 else "Monetary adds little beyond activity: traded value is already reflected in R and F clusters")

### 3.2 Written review

**Cluster profiles and sector composition.** Risk and return clusters (k = 3) separate mainly by beta: [A] is low-beta (median [ ]), [C] high-beta (median [ ]), with median weekly returns of [ ] to [ ]. Activity clusters (k = 4) separate by Recency, Frequency and traded value: [names], with median Monetary from $[ ]bn to $[ ]bn. The cross-tabulation shows [weak] overlap (adjusted Rand index [ ]), so beta and trading activity describe different dimensions. The high-beta cluster is concentrated in [sectors] ([ ]% against [ ]% of the index); the low-beta cluster in [sectors]. The high-Monetary activity cluster is [sector pattern]. Silhouette scores of [ ] indicate moderate separation, so boundaries are fuzzy.

**Sensitivity findings and business uses.** Recency and Frequency correlate at [ ]; log1p Monetary correlates weakly with both ([ ]). The first principal component explains [ ]% of variance, so the activity features partly overlap and can give trading activity extra influence even after standardisation; correlation alone does not prove dominance. Removing Monetary gave an adjusted Rand index of [ ]; variance in traded value explained by clusters fell from [ ] to [ ], so Monetary mainly separates [liquidity levels]. Uses: monitoring shares with recent, frequent volume spikes for event review; tracking cluster migration between observation dates; checking portfolio exposure by beta cluster and sector; screening by traded value.

**Critical evaluation.** Weekly returns share common market and sector shocks, so observations are dependent and clusters partly reflect these shocks; beta is window-sensitive. Adjusted Close includes dividends, whereas Monetary uses unadjusted Close, so price bases differ. Using current constituents creates survivorship bias, overstating returns and understating failure. Recency, Frequency and Monetary overlap. Close times Volume ignores spread, depth and price impact, and scales with company size. The clusters describe 2021 to 2024 in-sample behaviour. They provide no evidence of future returns or reduced portfolio risk; that needs out-of-sample stability tests and costed portfolio backtests.